<a href="https://colab.research.google.com/github/adalves-ufabc/2026.Q3-PLN/blob/main/2026_Q3_PLN_AULA_13_Notebook_25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Processamento de Linguagem Natural [2026-Q3]**
Prof. Alexandre Donizeti Alves

## **LangChain [Dados Estruturados]**
---



In [ ]:
#@title Instalando o pacote LangChain
!pip install langchain

In [ ]:
#@title Versão do LangChain

import langchain

print(langchain.__version__)

1.4.7


In [ ]:
#@title Integração com o pacote da OpenAI

!pip install langchain-openai

In [ ]:
#@title Definindo a chave da API da OpenAI
from getpass import getpass

OPENAI_API_KEY = getpass()

··········


## **Dados Estruturados**

**Dados estruturados** são dados que são organizados e formatados de maneira sistemática, permitindo fácil acesso, análise e manipulação.

Frequentemente é útil ter um modelo que retorne uma saída que corresponda a um esquema específico.

**Método `.with_structured_output()`**

Esta é a maneira mais fácil e confiável de obter saídas estruturadas. O método `with_structured_output()` é implementado para modelos que fornecem APIs nativas para estruturar saídas, como chamadas de ferramentas/funções ou modo JSON, e utiliza essas capacidades internamente.

>

Este método recebe um esquema como entrada, que especifica os nomes, tipos e descrições dos atributos de saída desejados. O método retorna um `Runnable` semelhante a um modelo, exceto que, em vez de produzir strings ou Mensagens, ele produz objetos correspondentes ao esquema fornecido. O esquema pode ser especificado como uma classe `TypedDict`, um `JSON Schema` ou uma classe `Pydantic`. Se `TypedDict` ou `JSON Schema` forem usados, então um dicionário será retornado pelo `Runnable`, e se uma classe `Pydantic` for usada, então um objeto `Pydantic` será retornado.

>

Como exemplo, vamos fazer com que um modelo gere uma piada e separe a preparação do desfecho da piada:

In [ ]:
from langchain_openai import ChatOpenAI

modelo = ChatOpenAI(model="gpt-6-luna", api_key = OPENAI_API_KEY)

**Classe `Pydantic`**

Se quisermos que o modelo retorne um objeto `Pydantic`, basta passar a classe `Pydantic` desejada. A principal vantagem de usar `Pydantic` é que a saída gerada pelo modelo será validada. O `Pydantic` levantará um erro se algum campo obrigatório estiver faltando ou se algum campo for do tipo errado.

In [ ]:
from typing import Optional

from pydantic import BaseModel, Field

# Pydantic
class Piada(BaseModel):
    """Piada para contar ao usuário."""

    introducao: str = Field(description="A introdução da piada")
    arremate: str = Field(description="O desfecho da piada")
    avaliacao: Optional[int] = Field(
        default=None, description="Quão engraçada é a piada, de 1 a 10"
    )

In [ ]:
modelo_estruturado = modelo.with_structured_output(Piada)

modelo_estruturado.invoke("Me conte uma piada sobre gatos")

Piada(introducao='Por que o gato sentou em cima do computador?', arremate='Para ficar de olho no mouse!', avaliacao=7)

Além da estrutura da classe `Pydantic`, o nome da classe, a *docstring* e os nomes e as descrições fornecidas dos parâmetros são muito importantes. Na maioria das vezes, `with_structured_output` está usando a API de chamada de funções/ferramentas de um modelo, e você pode pensar efetivamente em todas essas informações como sendo adicionadas ao *prompt* do modelo.

**`TypedDict` ou `JSON Schema`**

Se você não quiser usar `Pydantic`, explicitamente não quiser validação dos argumentos ou quiser ser capaz de transmitir as saídas do modelo, pode definir seu esquema usando uma classe `TypedDict`. Opcionalmente, podemos usar uma sintaxe especial `Annotated` suportada pelo **LangChain**, que permite especificar o valor padrão e a descrição de um campo. Observe que o valor padrão não é preenchido automaticamente se o modelo não o gerar; ele é usado apenas na definição do esquema que é passado para o modelo.

In [ ]:
from typing_extensions import Annotated, TypedDict

# TypedDict
class Piada(TypedDict):
    """Piada para contar ao usuário."""

    introducao: Annotated[str, ..., "introducao"]
    arremate: Annotated[str, ..., "O desfecho da piada"]
    avaliacao: Annotated[Optional[int], None, "Quão engraçada é a piada, de 1 a 10"]

In [ ]:
modelo_estruturado = modelo.with_structured_output(Piada)

modelo_estruturado.invoke("Me conte uma piada sobre gatos")

{'introducao': 'Por que o gato sentou em cima do computador?',
 'arremate': 'Para ficar de olho no mouse!',
 'avaliacao': 7}

De maneira equivalente, podemos passar um dicionário `JSON Schema`. Isso não requer importações ou classes e torna muito claro como cada parâmetro é documentado, com o custo de ser um pouco mais verboso.

In [ ]:
json_schema = {
    "title": "piada",
    "description": "Piada para contar ao usuário.",
    "type": "object",
    "properties": {
        "introducao": {
            "type": "string",
            "description": "A introdução da piada",
        },
        "arremate": {
            "type": "string",
            "description": "O desfecho da piada",
        },
        "avaliacao": {
            "type": "integer",
            "description": "Quão engraçada é a piada, de 1 a 10",
            "default": None,
        },
    },
    "required": ["introducao", "arremate"],
}


In [ ]:
modelo_estruturado = modelo.with_structured_output(json_schema)

modelo_estruturado.invoke("Me conte uma piada sobre gatos")

{'introducao': 'Por que o gato foi ao computador?',
 'arremate': 'Para entrar no mouse!',
 'avaliacao': 6}

Outro exemplo:

In [ ]:
class Evento(BaseModel):
    nome: str = Field(description="Nome do evento")
    local: str = Field(description="Local de realização")
    data: str = Field(description="Data do evento")
    horario: str = Field(description="Horário de início")

modelo_estruturado = modelo.with_structured_output(Evento)

prompt = ChatPromptTemplate.from_messages([
    ("system", """Extraia as informações do evento mencionado pelo usuário.
    Retorne o nome, o local, a data e o horário do evento."""),
    ("human", "{input}")
])

cadeia = prompt | modelo_estruturado

resposta = cadeia.invoke({
    "input": "A palestra sobre inteligência artificial será realizada na UFABC no dia 15 de novembro de 2026, às 19 horas."
})

In [ ]:
resposta

Evento(nome='Palestra sobre inteligência artificial', local='UFABC', data='15 de novembro de 2026', horario='19 horas')

In [ ]:
resposta.nome

'Palestra sobre inteligência artificial'

In [ ]:
resposta.local

'UFABC'

Mais um exemplo:

In [ ]:
class AnaliseSentimento(BaseModel):
    sentimento: str = Field(description="Positivo, negativo ou neutro")
    produto: str = Field(description="Produto avaliado")
    aspectos_positivos: list[str] = Field(description="Aspectos positivos mencionados")
    aspectos_negativos: list[str] = Field(description="Aspectos negativos mencionados")
    nota: int = Field(description="Nota estimada de 1 a 5")

modelo_estruturado = modelo.with_structured_output(AnaliseSentimento)

prompt = ChatPromptTemplate.from_messages([
    ("system", """Analise a avaliação de produto apresentada pelo usuário.
    Identifique o sentimento predominante, o produto, os aspectos positivos,
    os aspectos negativos e atribua uma nota de 1 a 5."""),
    ("human", "{input}")
])

cadeia = prompt | modelo_estruturado

resposta = cadeia.invoke({
    "input": "Comprei um fone Bluetooth da marca X. A bateria dura bastante e o som é excelente, mas o microfone apresenta ruídos durante as chamadas."
})

In [ ]:
resposta

AnaliseSentimento(sentimento='positivo', produto='Fone Bluetooth da marca X', aspectos_positivos=['A bateria dura bastante', 'O som é excelente'], aspectos_negativos=['O microfone apresenta ruídos durante as chamadas'], nota=4)

**Escolhendo entre múltiplos esquemas**

A maneira mais simples de permitir que o modelo escolha entre vários esquemas é criar um esquema "pai" que tenha um atributo do tipo `Union`:

In [ ]:
from typing import Union

# Pydantic

class Piada(BaseModel):
    """Piada para contar ao usuário."""

    introducao: str = Field(description="A introdução da piada")
    arremate: str = Field(description="O desfecho da piada")
    avaliacao: Optional[int] = Field(
        default=None, description="Quão engraçada é a piada, de 1 a 10"
    )

class RespostaConversacional(BaseModel):
    """Responda de maneira conversacional. Seja gentil e prestativo."""

    resposta: str = Field(description="Uma resposta conversacional à pergunta do usuário")

class Resposta(BaseModel):
    saida: Union[Piada, RespostaConversacional]

In [ ]:
modelo_estruturado = modelo.with_structured_output(Resposta)

resposta = modelo_estruturado.invoke("Me conte uma piada sobre gatos")

In [ ]:
resposta

Resposta(saida=Piada(introducao='Por que o gato foi ao computador?', arremate='Para navegar na rede e caçar uns *ratos*!', avaliacao=7))

In [ ]:
resposta.saida

Piada(introducao='Por que o gato foi ao computador?', arremate='Para navegar na rede e caçar uns *ratos*!', avaliacao=7)

In [ ]:
resposta.saida.introducao

'Por que o gato foi ao computador?'

In [ ]:
resposta = modelo_estruturado.invoke("Como você está hoje?")

In [ ]:
resposta

Resposta(saida=RespostaConversacional(resposta='Estou bem, obrigado por perguntar! E você, como está hoje?'))

In [ ]:
resposta.saida.resposta

'Estou bem, obrigado por perguntar! E você, como está hoje?'

***Few-shot prompting***

***Few-shot prompting*** é uma técnica onde se fornecem alguns exemplos específicos no *prompt* para guiar o modelo na geração de respostas. Em vez de treinar o modelo com uma nova base de dados completa, você apenas inclui alguns exemplos relevantes diretamente no *prompt*.

Para esquemas mais complexos, é muito útil adicionar exemplos ao *prompt*.

A maneira mais simples e universal é adicionar exemplos a uma mensagem do sistema no *prompt*:

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

sistema = """Você é um comediante hilário. Sua especialidade são piadas do tipo "knock-knock".
Retorne uma piada que tenha a introdução (a resposta para "Quem está aí?") e o arremate final (a resposta para "<introdução> quem?").

Aqui estão alguns exemplos de piadas:

exemplo_usuario: Me conte uma piada sobre gatos
exemplo_assistente: {{"introducao": "Por que os gatos não jogam cartas?", "arremate": "Porque eles têm medo dos cães que podem trapacear!", "avaliacao": 8}}

exemplo_usuario: Conte outra piada sobre cães
exemplo_assistente: {{"introducao": "Por que os cães levam uma bola para a escola?", "arremate": "Porque eles querem aprender a jogar!", "ratavaliacaoing": 7}}

exemplo_usuario: Agora sobre peixes
exemplo_assistente: {{"introducao": "Por que o peixe foi ao banco?", "arremate": "Para abrir uma conta corrente!", "avaliacao": 9}}"""

# template de prompt
prompt = ChatPromptTemplate.from_messages([("system", sistema), ("human", "{input}")])

# uso do prompt com um modelo estruturado
modelo_estruturado_few_shot = prompt | modelo_estruturado

modelo_estruturado_few_shot.invoke("Me conte uma piada sobre elefantes")

Resposta(saida=Piada(introducao='Tromba', arremate='Tromba quem? O elefante, que bateu à porta e ficou preso na entrada!', avaliacao=7))

Outro exemplo:

In [ ]:
# Pydantic

class Produto(BaseModel):
    """Descrição de um produto."""

    nome: str = Field(description="O nome do produto")
    descricao: str = Field(description="A descrição do produto")

In [ ]:
modelo_estruturado = modelo.with_structured_output(Produto)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

sistema = """Você é um redator de descrições de produtos experiente. Sua tarefa é criar descrições detalhadas e envolventes para produtos.
Cada descrição deve incluir informações sobre as principais características e benefícios do produto.

Aqui estão alguns exemplos de descrições:

exemplo_usuario: Descreva um smartphone moderno
exemplo_assistente: {{"nome": "Smartphone UltraX", "descricao": "O Smartphone UltraX é equipado com uma tela AMOLED de 6,7 polegadas, processador octa-core de última geração e câmera tripla de 64MP. Oferece um desempenho rápido e uma experiência de usuário excepcional com bateria de longa duração e carregamento rápido."}}

exemplo_usuario: Descreva um aspirador de pó robô
exemplo_assistente: {{"nome": "Aspirador Robô CleanPro", "descricao": "O Aspirador Robô CleanPro é ideal para manter sua casa limpa sem esforço. Com tecnologia de navegação inteligente, ele limpa todos os cantos e vem com um sistema de filtragem HEPA que captura alérgenos e poeira. Compacto e silencioso, é fácil de usar e programar."}}

exemplo_usuario: Agora sobre uma cafeteira
exemplo_assistente: {{"nome": "Cafeteira Espresso Elite", "descricao": "A Cafeteira Espresso Elite oferece uma experiência de café de qualidade barista no conforto de sua casa. Com um sistema de pressão de 15 bar, ela extrai o máximo sabor dos grãos. Possui um moinho integrado e controle de temperatura preciso para preparar o café perfeito a cada vez."}}"""

# template de prompt
prompt = ChatPromptTemplate.from_messages([("system", sistema), ("human", "{input}")])

# uso do prompt com um modelo estruturado
modelo_estruturado_few_shot = prompt | modelo_estruturado

modelo_estruturado_few_shot.invoke("Descreva um tablet moderno")

Produto(nome='Tablet NovaTab Pro', descricao='O Tablet NovaTab Pro combina desempenho e versatilidade em um design fino e leve. Sua tela de alta resolução oferece imagens nítidas e cores vibrantes para assistir a vídeos, ler e navegar com conforto. Equipado com processador rápido, armazenamento amplo e bateria de longa duração, ele facilita o trabalho, os estudos e o entretenimento ao longo do dia. Compatível com acessórios como teclado e caneta digital, adapta-se a diferentes tarefas com praticidade.')

Para modelos que suportam mais de um meio de estruturar saídas, você pode especificar qual método usar com o argumento `method=`.

In [ ]:
modelo_estruturado = modelo.with_structured_output(None, method="json_mode")

modelo_estruturado.invoke(
    "Descreva um tablet moderno, responda em JSON com as chaves `nome` e `descricao`"
)

{'nome': 'Tablet moderno',
 'descricao': 'Dispositivo portátil com tela sensível ao toque de alta resolução, design leve e fino, desempenho rápido, bateria de longa duração e conectividade sem fio. Ideal para estudar, trabalhar, assistir a vídeos, ler e navegar na internet.'}

Mais um exemplo:

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

class Viagem(BaseModel):
    destino: str = Field(description="Destino da viagem")
    duracao_dias: int = Field(description="Duração da viagem em dias")
    orcamento_reais: float = Field(description="Orçamento máximo em reais")

modelo_estruturado = modelo.with_structured_output(Viagem)

In [ ]:
sistema = """Você é um assistente especializado em planejamento de viagens.
Extraia dos pedidos do usuário o destino, a duração da viagem e o orçamento máximo.
Retorne os dados no formato estruturado solicitado.

Exemplos:

exemplo_usuario: Quero viajar para Paris por 7 dias e gastar até 5000 reais.
exemplo_assistente: {{"destino": "Paris", "duracao_dias": 7, "orcamento_reais": 5000}}

exemplo_usuario: Pretendo passar 10 dias em Lisboa com orçamento de 8000 reais.
exemplo_assistente: {{"destino": "Lisboa", "duracao_dias": 10, "orcamento_reais": 8000}}

exemplo_usuario: Quero conhecer Buenos Aires durante 5 dias, gastando no máximo 3000 reais.
exemplo_assistente: {{"destino": "Buenos Aires", "duracao_dias": 5, "orcamento_reais": 3000}}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", sistema),
    ("human", "{input}")
])

modelo_estruturado_few_shot = prompt | modelo_estruturado

resposta = modelo_estruturado_few_shot.invoke({
    "input": "Quero conhecer Roma por 8 dias e tenho 6000 reais disponíveis."
})

In [ ]:
resposta

Viagem(destino='Roma', duracao_dias=8, orcamento_reais=6000.0)